In [1]:
import torch
from reasoning_from_scratch.ch02 import get_device
from reasoning_from_scratch.ch03 import load_model_and_tokenizer
from IPython.display import Latex, display
import time
import numpy as np

device = get_device()
# device = torch.device("cpu")

model, tokenizer = load_model_and_tokenizer(
    which_model="base",
    device=device,
    use_compile=False
)

Using NVIDIA CUDA GPU
✓ qwen3/qwen3-0.6B-base.pth already up-to-date


## Stephen RL Hacking 5

- https://github.com/rasbt/reasoning-from-scratch/tree/main/ch06
- https://github.com/McGill-NLP/nano-aha-moment/blob/main/nano_r1.ipynb
- https://github.com/rasbt/reasoning-from-scratch/blob/main/ch07/01_main-chapter-code/ch07_main.ipynb
- Big Illustrator file and jupyter notebook(s) I think is the vibe here
- aha moment results
- a breakdown of gradient math to achieve these results
- Comparison to supervised learning
- comparison of the different Os
- Connection to Karpathy’s post

In [2]:
from reasoning_from_scratch.ch03 import render_prompt
from reasoning_from_scratch.ch04 import (
    generate_text_stream_concat_flex,
    generate_text_top_p_stream_cache
)

raw_prompt = (
    "Half the value of $3x-9$ is $x+37$. "
    "What is the value of $x$?"
)
prompt = render_prompt(raw_prompt)

torch.manual_seed(0)
response = generate_text_stream_concat_flex(
    model, tokenizer, prompt, device,
    max_new_tokens=2048, verbose=True,
    generate_func=generate_text_top_p_stream_cache,
    temperature=0.9,
    top_p=0.9
)

 46

In [3]:
import json
import requests
from pathlib import Path

def load_math_train(local_path="math_train.json", save_copy=True):
    local_path = Path(local_path)
    url = (
        "https://raw.githubusercontent.com/rasbt/"
        "math_full_minus_math500/refs/heads/main/"
        "math_full_minus_math500.json"
    )

    if local_path.exists():
        with local_path.open("r", encoding="utf-8") as f:
            data = json.load(f)
    else:
        r = requests.get(url, timeout=30)
        r.raise_for_status()
        data = r.json()

        if save_copy:  # Saves a local copy
            with local_path.open("w", encoding="utf-8") as f:
                json.dump(data, f, indent=2)

    return data

In [4]:
math_train = load_math_train()
print("Dataset size:", len(math_train))

Dataset size: 12000


In [5]:
from reasoning_from_scratch.qwen3 import KVCache
from reasoning_from_scratch.ch04 import top_p_filter


@torch.no_grad()
def sample_response(
    model,
    tokenizer,
    prompt,
    device,
    max_new_tokens=512,
    temperature=0.8,
    top_p=0.9,
):
    input_ids = torch.tensor(
        tokenizer.encode(prompt),
        device=device
        )

    cache = KVCache(n_layers=model.cfg["n_layers"])
    model.reset_kv_cache()
    logits = model(input_ids.unsqueeze(0), cache=cache)[:, -1]

    generated = []
    for _ in range(max_new_tokens):
        if temperature and temperature != 1.0:
            logits = logits / temperature

        probas = torch.softmax(logits, dim=-1)
        probas = top_p_filter(probas, top_p)
        next_token = torch.multinomial(
            probas.cpu(), num_samples=1
        ).to(device)

        if (
            tokenizer.eos_token_id is not None
            and next_token.item() == tokenizer.eos_token_id
        ):
            break
        generated.append(next_token.item())
        logits = model(next_token, cache=cache)[:, -1]

    full_token_ids = torch.cat(
        [input_ids,
         torch.tensor(generated, device=device, dtype=input_ids.dtype),]
    )
    return full_token_ids, input_ids.numel(), tokenizer.decode(generated)

### Sampling before training

In [6]:
torch.manual_seed(0)

raw_prompt = (
    "Half the value of $3x-9$ is $x+37$. "
    "What is the value of $x$?"
)
prompt = render_prompt(raw_prompt)

token_ids, prompt_len, answer_text = sample_response(
            model=model,
            tokenizer=tokenizer,
            prompt=prompt,
            device=device,
            max_new_tokens=512,
            temperature=0.9,
            top_p=0.9,
        )

print(answer_text)

 46


### Sampling after training

In [7]:
# from reasoning_from_scratch.qwen3 import download_qwen3_grpo_checkpoints
# download_qwen3_grpo_checkpoints(grpo_type="no_kl", step="00050")

In [8]:
model2, tokenizer = load_model_and_tokenizer(
    which_model="base",
    device=device,
    use_compile=False
)

state_dict = torch.load('qwen3-0.6B-rlvr-grpo-step00050.pth', map_location="cpu")
model2.load_state_dict(state_dict)
model2.to(device)

✓ qwen3/qwen3-0.6B-base.pth already up-to-date


Qwen3Model(
  (tok_emb): Embedding(151936, 1024)
  (trf_blocks): ModuleList(
    (0-27): 28 x TransformerBlock(
      (att): GroupedQueryAttention(
        (W_query): Linear(in_features=1024, out_features=2048, bias=False)
        (W_key): Linear(in_features=1024, out_features=1024, bias=False)
        (W_value): Linear(in_features=1024, out_features=1024, bias=False)
        (out_proj): Linear(in_features=2048, out_features=1024, bias=False)
        (q_norm): RMSNorm()
        (k_norm): RMSNorm()
      )
      (ff): FeedForward(
        (fc1): Linear(in_features=1024, out_features=3072, bias=False)
        (fc2): Linear(in_features=1024, out_features=3072, bias=False)
        (fc3): Linear(in_features=3072, out_features=1024, bias=False)
      )
      (norm1): RMSNorm()
      (norm2): RMSNorm()
    )
  )
  (final_norm): RMSNorm()
  (out_head): Linear(in_features=1024, out_features=151936, bias=False)
)

In [9]:
torch.manual_seed(0)

raw_prompt = (
    "Half the value of $3x-9$ is $x+37$. "
    "What is the value of $x$?"
)
prompt = render_prompt(raw_prompt)

token_ids, prompt_len, answer_text = sample_response(
            model=model2,
            tokenizer=tokenizer,
            prompt=prompt,
            device=device,
            max_new_tokens=512,
            temperature=0.9,
            top_p=0.9,
        )

display(Latex(answer_text))

<IPython.core.display.Latex object>

## Now let's get into gradient descent mechanics on a real example

- Pull in support functions that I will ultimately unpack

In [10]:
from reasoning_from_scratch.ch03 import (
    extract_final_candidate, grade_answer
)

def reward_rlvr(answer_text, ground_truth):
    extracted = extract_final_candidate(
        answer_text, fallback=None  # Require \boxed{}
    )
    if not extracted:
        return 0.0
    correct = grade_answer(extracted, ground_truth)
    return float(correct)

In [11]:
@torch.inference_mode()
def avg_logprob_answer(model, tokenizer, prompt, answer, device="cpu"):

    # Encode prompt and answer tokens separately to get the prompt length later
    prompt_ids = tokenizer.encode(prompt)
    answer_ids = tokenizer.encode(answer)
    full_ids = torch.tensor(prompt_ids + answer_ids, device=device)

    # Same as in calc_next_token_logprobas before
    logits = model(full_ids.unsqueeze(0)).squeeze(0)
    logprobs = torch.log_softmax(logits, dim=-1)

    # Index range for positions corresponding to answer tokens
    start = len(prompt_ids) - 1
    end = full_ids.shape[0] - 1

    # Same as before, except for using start and end
    t_idx = torch.arange(start, end, device=device)
    next_tokens = full_ids[start + 1 : end + 1]
    next_token_logps = logprobs[t_idx, next_tokens]

    # Average over the answer token scores
    return torch.mean(next_token_logps).item()

#SW - maybe work wiht this more robust version from Raschka? Renaming here
def sequence_logprob(model, token_ids, prompt_len):
    logits = model(token_ids.unsqueeze(0)).squeeze(0).float()
    logprobs = torch.log_softmax(logits, dim=-1)

    # Positions whose next-token probabilities we want
    # These correspond to predicting token_ids[t + 1] from position t
    start = prompt_len - 1
    end = token_ids.shape[0] - 1

    t_idx = torch.arange(start, end, device=token_ids.device)
    next_tokens = token_ids[start + 1 : end + 1]
    next_token_logps = logprobs[t_idx, next_tokens]

    # Sum log-probabilities over the answer tokens
    return torch.sum(next_token_logps)


def compute_grpo_loss(
    model,
    tokenizer,
    example,
    device,
    num_rollouts=2,
    max_new_tokens=256,
    temperature=0.8,
    top_p=0.9,
):
    assert num_rollouts >= 2
    roll_logps, roll_rewards, samples = [], [], []
    prompt = render_prompt(example["problem"])

    was_training = model.training
    model.eval()

    for _ in range(num_rollouts):
        # Stage 1: generate rollouts
        token_ids, prompt_len, text = sample_response(
            model=model,
            tokenizer=tokenizer,
            prompt=prompt,
            device=device,
            max_new_tokens=max_new_tokens,
            temperature=temperature,
            top_p=top_p,
        )
        # Stage 2: compute rewards
        reward = reward_rlvr(text, example["answer"])
        
        # Stage 4: compute logprobs
        logp = sequence_logprob(model, token_ids, prompt_len)

        roll_logps.append(logp)
        roll_rewards.append(reward)
        samples.append(
            {
                "text": text,
                "reward": reward,
                "gen_len": token_ids.numel() - prompt_len,
            }
        )

    if was_training:
        model.train()

    # Stage 2: collect all rewards
    rewards = torch.tensor(roll_rewards, device=device)

    # Stage 3: compute advantages
    advantages = (rewards - rewards.mean()) / (rewards.std() + 1e-4)

    # Stage 4: collect all logprobs
    logps = torch.stack(roll_logps)

    # Stage 5: compute policy gradient loss
    pg_loss = -(advantages.detach() * logps).mean()
    loss = pg_loss  # In the next chapter we add a KL term here

    return {
        "loss": loss.item(),
        "pg_loss": pg_loss.item(),
        "rewards": roll_rewards,
        "advantages": advantages.detach().cpu().tolist(),
        "samples": samples,
        "loss_tensor": loss,
    }


- Ok yeah let me deconstruct the training loop myself here
- Maybe I basically unpack Raschka's methods and make sure that I get the same answer each time?

In [12]:
num_rollouts=8
max_new_tokens=512
temperature=0.8
top_p=0.9
lr=1e-5

optimizer = torch.optim.AdamW(model.parameters(), lr=lr)

In [13]:
step=2

optimizer.zero_grad()

current_step = step + 1
example = math_train[step % len(math_train)]

In [14]:
Latex(example['problem'])

<IPython.core.display.Latex object>

In [15]:
example['problem']

'What is the degree of the polynomial $(4 +5x^3 +100 +2\\pi x^4 + \\sqrt{10}x^4 +9)$?'

In [16]:
example['answer']

'4'

In [38]:
roll_logps, roll_rewards, samples = [], [], []
prompt = render_prompt(example["problem"])

was_training = model.training
model.eval();

for i in range(num_rollouts):
    
    torch.manual_seed(24+i) #Reproducability, 8+ is not bad, 24+ is pretty nice
    # Stage 1: generate rollouts
    token_ids, prompt_len, text = sample_response(
        model=model,
        tokenizer=tokenizer,
        prompt=prompt,
        device=device,
        max_new_tokens=max_new_tokens,
        temperature=temperature,
        top_p=top_p,
    )
    # Stage 2: compute rewards
    reward = reward_rlvr(text, example["answer"])
    
    # Stage 4: compute logprobs
    logp = sequence_logprob(model, token_ids, prompt_len)

    roll_logps.append(logp)
    roll_rewards.append(reward)
    samples.append(
        {
            "text": text,
            "reward": reward,
            "gen_len": token_ids.numel() - prompt_len,
        }
    )

if was_training:
    model.train()

# Stage 2: collect all rewards
rewards = torch.tensor(roll_rewards, device=device)

# Stage 3: compute advantages
advantages = (rewards - rewards.mean()) / (rewards.std() + 1e-4)

# Stage 4: collect all logprobs
logps = torch.stack(roll_logps)

# Stage 5: compute policy gradient loss
pg_loss = -(advantages.detach() * logps).mean()
loss = pg_loss  # In the next chapter we add a KL term here

In [40]:
rewards

tensor([0., 0., 0., 1., 0., 1., 0., 0.], device='cuda:0')

In [41]:
advantages

tensor([-0.5399, -0.5399, -0.5399,  1.6198, -0.5399,  1.6198, -0.5399, -0.5399],
       device='cuda:0')

In [42]:
logps

tensor([ -2.5370,  -3.3012,  -3.3012, -13.7855,  -2.5370, -25.8830,  -3.5895,
         -3.3012], device='cuda:0', grad_fn=<StackBackward0>)

In [55]:
advantages.detach() * logps

tensor([  1.3699,   1.7824,   1.7824, -22.3303,   1.3699, -41.9262,   1.9381,
          1.7824], device='cuda:0', grad_fn=<MulBackward0>)

In [56]:
-(advantages.detach() * logps).mean()

tensor(6.7789, device='cuda:0', grad_fn=<NegBackward0>)

In [44]:
samples

[{'text': ' 5', 'reward': 0.0, 'gen_len': 2},
 {'text': ' The degree of the polynomial is 4.', 'reward': 0.0, 'gen_len': 9},
 {'text': ' The degree of the polynomial is 4.', 'reward': 0.0, 'gen_len': 9},
 {'text': ' To find the degree of the polynomial, we need to identify the term with the highest power of $x$.\nIn this case, the term with the highest power of $x$ is $2\\pi x^4$, which has a power of 4.\nTherefore, the degree of the polynomial is $\\boxed{4}$.',
  'reward': 1.0,
  'gen_len': 68},
 {'text': ' 5', 'reward': 0.0, 'gen_len': 2},
 {'text': " To find the degree of the polynomial, we need to identify the term with the highest power of $x$.\n\nThe given polynomial is $(4 +5x^3 +100 +2\\pi x^4 + \\sqrt{10}x^4 +9)$.\n\nFirst, let's combine like terms:\n\n$$(4 +100 +9) + (5x^3) + (2\\pi x^4 + \\sqrt{10}x^4)$$\n\n$$113 + 5x^3 + (2\\pi + \\sqrt{10})x^4$$\n\nNow, we can see that the term with the highest power of $x$ is $(2\\pi + \\sqrt{10})x^4$.\n\nTherefore, the degree of the pol

In [46]:
Latex(samples[3]['text'])

<IPython.core.display.Latex object>

In [47]:
Latex(samples[5]['text'])

<IPython.core.display.Latex object>

In [48]:
Latex(samples[6]['text'])

<IPython.core.display.Latex object>

In [50]:
Latex(samples[7]['text'])

<IPython.core.display.Latex object>

In [54]:
roll_logps

[tensor(-2.5370, device='cuda:0', grad_fn=<SumBackward0>),
 tensor(-3.3012, device='cuda:0', grad_fn=<SumBackward0>),
 tensor(-3.3012, device='cuda:0', grad_fn=<SumBackward0>),
 tensor(-13.7855, device='cuda:0', grad_fn=<SumBackward0>),
 tensor(-2.5370, device='cuda:0', grad_fn=<SumBackward0>),
 tensor(-25.8830, device='cuda:0', grad_fn=<SumBackward0>),
 tensor(-3.5895, device='cuda:0', grad_fn=<SumBackward0>),
 tensor(-3.3012, device='cuda:0', grad_fn=<SumBackward0>)]

In [83]:
logp

tensor(-5.5586, device='cuda:0', grad_fn=<SumBackward0>)

In [51]:
sequence_logprob(model, token_ids, prompt_len)

tensor(-3.3012, device='cuda:0', grad_fn=<SumBackward0>)

In [52]:
logits = model(token_ids.unsqueeze(0)).squeeze(0).float()
logprobs = torch.log_softmax(logits, dim=-1)

In [53]:
len(tokenizer.encode(prompt)), logits.shape, logprobs.shape #Includes question and response

(70, torch.Size([79, 151936]), torch.Size([79, 151936]))

In [92]:
# Positions whose next-token probabilities we want
# These correspond to predicting token_ids[t + 1] from position t
start = prompt_len - 1
end = token_ids.shape[0] - 1

In [93]:
start

143

In [94]:
end #Ok yeah sure!

149

In [95]:
t_idx = torch.arange(start, end, device=token_ids.device)

In [96]:
t_idx

tensor([143, 144, 145, 146, 147, 148], device='cuda:0')

In [97]:
next_tokens = token_ids[start + 1 : end + 1]

In [104]:
tokenizer.decode(next_tokens.tolist())

' \\boxed{64}'

In [105]:
next_token_logps = logprobs[t_idx, next_tokens]

In [106]:
next_token_logps

tensor([-0.5423, -0.1268, -0.0085, -2.7693, -1.7801, -0.3316], device='cuda:0',
       grad_fn=<IndexBackward0>)

In [107]:
torch.exp(next_token_logps)

tensor([0.5814, 0.8809, 0.9915, 0.0627, 0.1686, 0.7178], device='cuda:0',
       grad_fn=<ExpBackward0>)

Let me gut check this real quick:

In [112]:
probs = torch.nn.Softmax(-1)(logits)

In [117]:
probs[t_idx, next_tokens]

tensor([0.5814, 0.8809, 0.9915, 0.0627, 0.1686, 0.7178], device='cuda:0',
       grad_fn=<IndexBackward0>)

Ok yeah cool, then one more sanity check here

In [118]:
logprobs.shape, probs.shape

(torch.Size([150, 151936]), torch.Size([150, 151936]))

In [130]:
for i in range(start, end):
    print(i, np.exp(logprobs[i].max().item()), probs[i].max().item())

143 0.5813920140786188 0.5813920497894287
144 0.8809390303559367 0.8809390068054199
145 0.9915288814700534 0.9915288686752319
146 0.40890876842257345 0.40890875458717346
147 0.21650828758238583 0.21650829911231995
148 0.7177840580727645 0.71778404712677


- Ok indexing feels slightly funky -> like we should go one more forward - buuuut other than that this makes a ton of sense. 

In [131]:
torch.sum(next_token_logps)

tensor(-5.5586, device='cuda:0', grad_fn=<SumBackward0>)

Ok yeah this feels like no problem! We're just taking the sum of the answer logprobs -> easy peasy! There's a broader question of why -> which definitely ties back (potentially) to karpathy's derivation! But other that that nuance, this seems fine!

...How confident is the model in it's answer...in log space...

In [137]:
torch.exp(torch.sum(next_token_logps)) #I think we can interpret this as the probability of the full answer (multipling probs)

tensor(0.0039, device='cuda:0', grad_fn=<ExpBackward0>)

In [139]:
probs[t_idx, next_tokens].prod() #Yep ok cool. 

tensor(0.0039, device='cuda:0', grad_fn=<ProdBackward0>)

- Ok this doesn't seem to bad!!!
- In Ch7 Raschka talks about PPO too it looks like!
- NICE

- In mathematical notation, we can write the policy gradient loss as follows:

$$\mathcal{L}_{\mathrm{PG}}
= -\frac{1}{N} \sum_{i=1}^{N} A_i \sum_{t=1}^{T_i} \log p_W\!\left( y_t^{(i)} \mid y_{<t}^{(i)}, x^{(i)} \right)$$

- $N$ denotes the number of rollouts in the batch
- $y_1^{(i)}, ..., y_{T_i}^{(i)}$ are the tokens of the $i$-th generated response of length $T_i$
- $y_{<t}^{(i)}$ represents all previously generated tokens in that response
- $x^{(i)}$ is the corresponding input prompt for the $i$-th rollout
- $p_W$ denotes the model's policy, that is, the probability distribution over next tokens parameterized by the weights $W$
- $A_i$ is the advantage assigned to the full $i$-th rollout
- The inner sum computes the sequence-level log-probability of a rollout
- The outer average computes advantage-weighted log-probabilities across rollouts

- What is the equation for what I have so far?
- Why is this the thing to actually optimize? What happens to our model when we actually take the derivative of this expression? How does this compare to supervised cross entropy?
- What’s the connection to Karpathy?
- How does what I have so far fit with the big gross GRPO equation?

- Ok feeling pretty good about the top 3 bullets there -> i need to get these results into a nice shareable format - that will take some work - but I think that things are connecting pretty well for me! I think the Nano Ah Moment notebook can actuallly help with (4), I think let's do that next, then back to building out illustrator!

In [35]:
logp

tensor(-5.5586, device='cuda:0', grad_fn=<SumBackward0>)

In [37]:
sequence_logprob(model, token_ids, prompt_len)

tensor(-5.5586, device='cuda:0', grad_fn=<SumBackward0>)

In [38]:
logits = model(token_ids.unsqueeze(0)).squeeze(0).float()
logprobs = torch.log_softmax(logits, dim=-1)

In [40]:
logits.shape

torch.Size([150, 151936])

In [42]:
logprobs.shape

torch.Size([150, 151936])

In [44]:
len(token_ids)

150

In [46]:
torch.log_softmax

<function torch._VariableFunctionsClass.log_softmax>

In [68]:
logprobs[-1].argmax(), logprobs[-1].max()

(tensor(151643, device='cuda:0'),
 tensor(-0.0346, device='cuda:0', grad_fn=<MaxBackward1>))

In [69]:
logits[-1].argmax(), logits[-1].max()

(tensor(151643, device='cuda:0'),
 tensor(25.3750, device='cuda:0', grad_fn=<MaxBackward1>))

In [74]:
torch.exp(logprobs[-1].max())

tensor(0.9660, device='cuda:0', grad_fn=<ExpBackward0>)

In [81]:
np.exp(-0.001), np.exp(-0.01), np.exp(-0.1), np.exp(-1), np.exp(-10)

(np.float64(0.999000499833375),
 np.float64(0.9900498337491681),
 np.float64(0.9048374180359595),
 np.float64(0.36787944117144233),
 np.float64(4.5399929762484854e-05))

In [61]:
Latex(tokenizer.decode([o.item() for o in token_ids.detach()]))

<IPython.core.display.Latex object>

In [30]:
samples

[{'text': ' \\boxed{98}', 'reward': 1.0, 'gen_len': 6},
 {'text': " To solve this problem, we need to find the largest number of band members \\( N \\) that satisfies the given conditions. Let's break down the problem step by step.\n\n### Step 1: Define the Variables\n- Let \\( m \\) be the number of band members in each row.\n- Let \\( r \\) be the number of rows.\n- The total number of band members is \\( N = m \\times r \\).\n- We are given that \\( N < 100 \\).\n\n### Step 2: Initial Formation\nWhen the band is formed, there are 2 members left over. This means:\n\\[\nN = m \\times r + 2\n\\]\nSince \\( N < 100 \\), we have:\n\\[\nm \\times r + 2 < 100\n\\]\n\\[\nm \\times r < 98\n\\]\n\n### Step 3: Modified Formation\nIf the director increases the number of members in each row by 1 and reduces the number of rows by 2, the new formation has exactly enough places for each band member. This means:\n\\[\nm \\times (r - 2) + 1 = m \\times r\n\\]\nSimplifying this equation:\n\\[\nm \\tim

In [31]:
advantages

tensor([ 1.4997, -0.4999, -0.4999, -0.4999], device='cuda:0')

In [32]:
rewards

tensor([1., 0., 0., 0.], device='cuda:0')

In [33]:
stats = compute_grpo_loss(
    model=model,
    tokenizer=tokenizer,
    example=example,
    device=device,
    num_rollouts=num_rollouts,
    max_new_tokens=max_new_tokens,
    temperature=temperature,
    top_p=top_p,
)

In [34]:
stats

{'loss': -0.0,
 'pg_loss': -0.0,
 'rewards': [0.0, 0.0, 0.0, 0.0],
 'advantages': [0.0, 0.0, 0.0, 0.0],
 'samples': [{'text': " To solve this problem, we need to find the largest number of band members that satisfies the given conditions. Let's break down the problem into smaller steps:\n\n1. Let \\( m \\) be the number of band members in each row, and \\( r \\) be the number of rows.\n2. The total number of band members \\( N \\) can be expressed as \\( N = m \\cdot r \\).\n3. According to the problem, when the band is arranged in a rectangular formation with \\( m \\) members in each row and \\( r \\) rows, there are 2 members left over. This can be written as:\n   \\[\n   N = m \\cdot r + 2\n   \\]\n4. When the band is arranged in a rectangular formation with \\( m+1 \\) members in each row and \\( r-2 \\) rows, there are exactly enough places for each band member. This can be written as:\n   \\[\n   N = (m+1) \\cdot (r-2)\n   \\]\n5. We need to find the largest \\( N \\) that satis

In [15]:


def train_rlvr_grpo(
    model,
    tokenizer,
    math_data,
    device,
    steps=None,
    num_rollouts=2,
    max_new_tokens=256,
    temperature=0.8,
    top_p=0.9,
    lr=1e-5,
    checkpoint_every=50,
    checkpoint_dir=".",
    csv_log_path=None,

):
    if steps is None:
        steps = len(math_data)

    # Stage 1: initialize optimize
    # (the model was already initialized outside the function)
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr)
    model.train()
    current_step = 0
    if csv_log_path is None:
        timestamp = time.strftime("%Y%m%d_%H%M%S")
        csv_log_path = f"train_rlvr_grpo_metrics_{timestamp}.csv"
    csv_log_path = Path(csv_log_path)

    try:
        # Stage 2: Iterate over training steps
        for step in range(steps):

            # Stage 3: Reset loss gradient
            # (it's best practice to do this at the beginning of each step)
            optimizer.zero_grad()

            current_step = step + 1
            example = math_data[step % len(math_data)]

            # Stage 4: calculate GRPO loss
            stats = compute_grpo_loss(
                model=model,
                tokenizer=tokenizer,
                example=example,
                device=device,
                num_rollouts=num_rollouts,
                max_new_tokens=max_new_tokens,
                temperature=temperature,
                top_p=top_p,
            )

            # Stage 5: Backward pass to calculate loss gradients
            stats["loss_tensor"].backward()

            # Clip large gradients to improve training stability
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)

            # Stage 6: Update model weights using loss gradients
            optimizer.step()

            # Stage 7: Collect rewards, response lengths, and losses
            reward_avg = torch.tensor(stats["rewards"]).mean().item()
            step_tokens = sum(
                sample["gen_len"] for sample in stats["samples"]
            )
            avg_response_len = (
                step_tokens / len(stats["samples"]) 
                if stats["samples"] else 0.0
            )
            append_csv_metrics(
                csv_log_path, current_step, steps, stats["loss"],
                reward_avg, avg_response_len,
            )

            # Print step metrics
            print(
                f"[Step {current_step}/{steps}] "
                f"loss={stats['loss']:.4f} "
                f"reward_avg={reward_avg:.3f} "
                f"avg_resp_len={avg_response_len:.1f}"
            )

            # Sample outputs (every 10 steps) to check if model
            # generates coherent text
            if current_step % 10 == 0:
                print(f"[Step {current_step}] sample outputs")
                for i, sample in enumerate(stats["samples"][:3]):
                    text = sample["text"].replace("\n", "\\n")
                    print(
                        f"  {i+1}) reward={sample['reward']:.3f} "
                        f"len={sample['gen_len']}: {text}"
                    )
                print()

            # Stage 8: Save model checkpoint
            if checkpoint_every and current_step % checkpoint_every == 0:
                ckpt_path = save_checkpoint(
                    model=model,
                    checkpoint_dir=checkpoint_dir,
                    step=current_step,
                )
                print(f"Saved checkpoint to {ckpt_path}")

    # Save a model checkpoint if we interrupt the training early
    except KeyboardInterrupt:
        ckpt_path = save_checkpoint(
            model=model,
            checkpoint_dir=checkpoint_dir,
            step=max(1, current_step),
            suffix="interrupt",
        )
        print(f"\nKeyboardInterrupt. Saved checkpoint to {ckpt_path}")
        return model

    return model


def save_checkpoint(model, checkpoint_dir, step, suffix=""):
    checkpoint_dir = Path(checkpoint_dir)
    checkpoint_dir.mkdir(parents=True, exist_ok=True)
    suffix = f"-{suffix}" if suffix else ""
    ckpt_path = (
        checkpoint_dir /
        f"qwen3-0.6B-rlvr-grpo-step{step:05d}{suffix}.pth"
    )
    torch.save(model.state_dict(), ckpt_path)
    return ckpt_path


def append_csv_metrics(
    csv_log_path,
    step_idx,
    total_steps,
    loss,
    reward_avg,
    avg_response_len,
):
    if not csv_log_path.exists():
        csv_log_path.write_text(
            "step,total_steps,loss,reward_avg,avg_response_len\n",
            encoding="utf-8",
        )
    with csv_log_path.open("a", encoding="utf-8") as f:
        f.write(
            f"{step_idx},{total_steps},{loss:.6f},{reward_avg:.6f},"
            f"{avg_response_len:.6f}\n"
        )

In [13]:
device = get_device()
model.to(device)

torch.manual_seed(0)

train_rlvr_grpo(
    model=model,
    tokenizer=tokenizer,
    math_data=math_train,
    device=device,
    steps=50,
    num_rollouts=4,
    max_new_tokens=512,
    temperature=0.8,
    top_p=0.9,
    lr=1e-5,
    checkpoint_every=5,
    checkpoint_dir=".",
    csv_log_path="train_rlvr_grpo_metrics.csv",
)

Using NVIDIA CUDA GPU
[Step 1/50] loss=-0.0000 reward_avg=0.000 avg_resp_len=5.2
[Step 2/50] loss=-0.0000 reward_avg=0.000 avg_resp_len=5.8
[Step 3/50] loss=3.5188 reward_avg=0.500 avg_resp_len=23.0
[Step 4/50] loss=-0.0000 reward_avg=0.000 avg_resp_len=1.5

KeyboardInterrupt. Saved checkpoint to qwen3-0.6B-rlvr-grpo-step00005-interrupt.pth


Qwen3Model(
  (tok_emb): Embedding(151936, 1024)
  (trf_blocks): ModuleList(
    (0-27): 28 x TransformerBlock(
      (att): GroupedQueryAttention(
        (W_query): Linear(in_features=1024, out_features=2048, bias=False)
        (W_key): Linear(in_features=1024, out_features=1024, bias=False)
        (W_value): Linear(in_features=1024, out_features=1024, bias=False)
        (out_proj): Linear(in_features=2048, out_features=1024, bias=False)
        (q_norm): RMSNorm()
        (k_norm): RMSNorm()
      )
      (ff): FeedForward(
        (fc1): Linear(in_features=1024, out_features=3072, bias=False)
        (fc2): Linear(in_features=1024, out_features=3072, bias=False)
        (fc3): Linear(in_features=3072, out_features=1024, bias=False)
      )
      (norm1): RMSNorm()
      (norm2): RMSNorm()
    )
  )
  (final_norm): RMSNorm()
  (out_head): Linear(in_features=1024, out_features=151936, bias=False)
)

In [14]:
# from reasoning_from_scratch.qwen3 import download_qwen3_grpo_checkpoints

# download_qwen3_grpo_checkpoints(grpo_type="no_kl", step="00050")

qwen3-0.6B-rlvr-grpo-step00050.pth: 100% (1433 MiB / 1433 MiB)


In [24]:
from IPython.display import Latex, display

In [23]:
model, tokenizer = load_model_and_tokenizer(
    which_model="base",
    device=device,
    use_compile=False
)

✓ qwen3/qwen3-0.6B-base.pth already up-to-date


In [17]:
model2, tokenizer = load_model_and_tokenizer(
    which_model="base",
    device=device,
    use_compile=False
)

state_dict = torch.load('qwen3-0.6B-rlvr-grpo-step00050.pth', map_location="cpu")
model2.load_state_dict(state_dict)
model2.to(device)

✓ qwen3/qwen3-0.6B-base.pth already up-to-date


Qwen3Model(
  (tok_emb): Embedding(151936, 1024)
  (trf_blocks): ModuleList(
    (0-27): 28 x TransformerBlock(
      (att): GroupedQueryAttention(
        (W_query): Linear(in_features=1024, out_features=2048, bias=False)
        (W_key): Linear(in_features=1024, out_features=1024, bias=False)
        (W_value): Linear(in_features=1024, out_features=1024, bias=False)
        (out_proj): Linear(in_features=2048, out_features=1024, bias=False)
        (q_norm): RMSNorm()
        (k_norm): RMSNorm()
      )
      (ff): FeedForward(
        (fc1): Linear(in_features=1024, out_features=3072, bias=False)
        (fc2): Linear(in_features=1024, out_features=3072, bias=False)
        (fc3): Linear(in_features=3072, out_features=1024, bias=False)
      )
      (norm1): RMSNorm()
      (norm2): RMSNorm()
    )
  )
  (final_norm): RMSNorm()
  (out_head): Linear(in_features=1024, out_features=151936, bias=False)
)

In [18]:
model2

device(type='cuda')

In [27]:
torch.manual_seed(0)

raw_prompt = (
    "Half the value of $3x-9$ is $x+37$. "
    "What is the value of $x$?"
)
prompt = render_prompt(raw_prompt)

token_ids, prompt_len, answer_text = sample_response(
            model=model,
            tokenizer=tokenizer,
            prompt=prompt,
            device=device,
            max_new_tokens=512,
            temperature=0.9,
            top_p=0.9,
        )

display(Latex(answer_text))

<IPython.core.display.Latex object>

In [28]:
torch.manual_seed(0)

raw_prompt = (
    "Half the value of $3x-9$ is $x+37$. "
    "What is the value of $x$?"
)
prompt = render_prompt(raw_prompt)

token_ids, prompt_len, answer_text = sample_response(
            model=model2,
            tokenizer=tokenizer,
            prompt=prompt,
            device=device,
            max_new_tokens=512,
            temperature=0.9,
            top_p=0.9,
        )

display(Latex(answer_text))

<IPython.core.display.Latex object>